In [ ]:
from src.analytics.observability_assets import write_observability_tables


def get_pipeline_id_from_api(pipeline_name: str = None):
    """Fetch pipeline ID from Databricks Workspace API by name or return the first pipeline."""
    from databricks.sdk import WorkspaceClient

    try:
        w = WorkspaceClient()
        if pipeline_name:
            # Use server-side filter for name matching
            pipelines = list(w.pipelines.list_pipelines(filter=f"name LIKE '%{pipeline_name}%'", max_results=1))
            if pipelines:
                return pipelines[0].pipeline_id
            return None
        else:
            # No filter: return the first pipeline
            pipelines = list(w.pipelines.list_pipelines(max_results=1))
            return pipelines[0].pipeline_id if pipelines else None
    except Exception as e:
        print(f"Note: Could not auto-fetch pipeline ID from API: {e}")
        return None


dbutils.widgets.text("catalog", "healthcare", "Catalog")
dbutils.widgets.text("analytics_schema", "analytics", "Analytics schema")
dbutils.widgets.text("pipeline_id", "", "Pipeline ID (fetched automatically if empty)")
dbutils.widgets.text("pipeline_name", "", "Pipeline name filter (optional)")
dbutils.widgets.text("published_event_log_table", "", "Published event-log table")

CATALOG = dbutils.widgets.get("catalog").strip()
ANALYTICS_SCHEMA = dbutils.widgets.get("analytics_schema").strip()
WIDGET_PIPELINE_ID = dbutils.widgets.get("pipeline_id").strip()
PIPELINE_NAME_FILTER = dbutils.widgets.get("pipeline_name").strip()
PUBLISHED_EVENT_LOG_TABLE = dbutils.widgets.get("published_event_log_table").strip()

# Try to fetch pipeline ID from API if not provided via widget
if WIDGET_PIPELINE_ID:
    PIPELINE_ID = WIDGET_PIPELINE_ID
    print(f"✓ Using widget pipeline ID: {PIPELINE_ID}")
else:
    API_PIPELINE_ID = get_pipeline_id_from_api(PIPELINE_NAME_FILTER or None)
    if API_PIPELINE_ID:
        PIPELINE_ID = API_PIPELINE_ID
        filter_msg = f" (filtered by name: '{PIPELINE_NAME_FILTER}')" if PIPELINE_NAME_FILTER else ""
        print(f"✓ Auto-fetched pipeline ID from API: {PIPELINE_ID}{filter_msg}")
    else:
        PIPELINE_ID = None
        if PIPELINE_NAME_FILTER:
            print(f"⚠ No pipeline found matching name: '{PIPELINE_NAME_FILTER}'")
        else:
            print("⚠ No pipelines found in workspace")

if PIPELINE_ID and PUBLISHED_EVENT_LOG_TABLE:
    raise ValueError("Provide only one of pipeline_id or published_event_log_table.")
if not PIPELINE_ID and not PUBLISHED_EVENT_LOG_TABLE:
    raise ValueError("Provide either pipeline_id, pipeline_name filter, or published_event_log_table. The pipeline_id is auto-fetched if left empty.")

print(f"Building observability assets into {CATALOG}.{ANALYTICS_SCHEMA}")

In [ ]:
persisted_tables = write_observability_tables(
    spark,
    pipeline_id=PIPELINE_ID or None,
    published_event_log_table=PUBLISHED_EVENT_LOG_TABLE or None,
    catalog=CATALOG,
    analytics_schema=ANALYTICS_SCHEMA,
)

display(
    spark.createDataFrame(
        [{"table_name": table_name, "status": status} for table_name, status in persisted_tables.items()]
    )
)

In [ ]:
display(spark.table(f"{CATALOG}.{ANALYTICS_SCHEMA}.ops_pipeline_updates"))
display(spark.table(f"{CATALOG}.{ANALYTICS_SCHEMA}.ops_expectation_metrics"))
display(spark.table(f"{CATALOG}.{ANALYTICS_SCHEMA}.ops_user_actions"))
display(spark.table(f"{CATALOG}.{ANALYTICS_SCHEMA}.ops_latest_failures"))